# Whisky clustering with Groovy

The data-science core of the classic BeakerX-era `Whiskey.ipynb` from
[groovy-data-science](https://github.com/paulk-asert/groovy-data-science),
ported to the `groovy-jupyter` kernel: `%%classpath add mvn` becomes `@Grab`
(session-wide), `%import` becomes plain imports (they persist across cells),
and session state uses binding-style variables.

Smile is pinned to 1.5.3 — the last Apache-2.0-licensed release. 86 Scotch
distilleries, 12 flavor dimensions, clustered with k-means. Expect cluster
sizes of 25/44/17 (cluster numbering may permute between runs).

In [1]:
@Grab('tech.tablesaw:tablesaw-core:0.36.0')
@Grab('com.github.haifengl:smile-core:1.5.3')
import tech.tablesaw.api.*
import smile.clustering.KMeans
'deps ready'

deps ready

In [2]:
records = Table.read().csv('/Users/paulk/Projects/groovy-data-science/subprojects/Whiskey/src/main/resources/whiskey.csv')
records.shape()

86 rows X 14 cols

In [3]:
features = records.columnNames() - ['RowID', 'Distillery']
data = (0..<records.rowCount()).collect { ri ->
    features.collect { f -> records.column(f).get(ri) as double } as double[]
} as double[][]
[data.length, data[0].length]

[86, 12]

In [4]:
kmeans = KMeans.lloyd(data, 3)
clusters = kmeans.clusterLabel.toList()
clusters.countBy { it }

{0=25, 2=44, 1=17}

In [5]:
names = records.column('Distillery')
grouped = (0..<clusters.size()).groupBy { clusters[it] }
grouped.collectEntries { k, v -> [k, v.take(5).collect { names.get(it) }] }

{0=[Aberfeldy, Aberlour, Auchroisk, Balmenach, Belvenie], 2=[AnCnoc, Ardmore, ArranIsleOf, Auchentoshan, Aultmore], 1=[Ardbeg, Balblair, Bowmore, Bruichladdich, Caol Ila]}